# Exp01 — Deep Learning-based Fault Detection (Local)

**연구 주제**: IndPenSim 공정 변수 기반 딥러닝 이상 감지  
**방법**: MSPC(PCA 기반, 기준선) vs LSTM Autoencoder  
**평가 지표**: AUROC, AUPRC, ROC Curve  
**실행 환경**: 로컬 (Jupyter Notebook / VS Code)

---
**실험 순서**
1. 환경 설정 및 데이터 로드
2. 전처리 (변수 선택, 정규화, 슬라이딩 윈도우)
3. 기준선: MSPC (PCA 기반 T² & SPE)
4. LSTM Autoencoder 학습
5. 이상 점수 산출 및 평가
6. 결과 비교 및 시각화

## 1. 환경 설정 및 데이터 로드

In [ ]:
import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

sys.path.append('..')  # src/ 경로 인식
from src.preprocessing import load_data, get_batch_sequences, train_test_split, make_windows, ONLINE_VARS
from src.models import LSTMAutoencoder
from src.trainer import train, evaluate, find_threshold

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

DATA_DIR    = Path('../data/raw')
CONFIG_PATH = Path('../configs/fault_detection.yaml')
RESULT_DIR  = Path('../results')

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
WINDOW_SIZE = cfg['data']['window_size']
STRIDE_TR   = cfg['data']['stride_train']
STRIDE_EV   = cfg['data']['stride_eval']

print('Device     :', DEVICE)
print('Data dir   :', DATA_DIR.resolve())
print('Config     :', cfg)

In [ ]:
df, df_stats = load_data(DATA_DIR)
print('Main data shape :', df.shape)
print('Stats shape     :', df_stats.shape)

## 2. 전처리

In [ ]:
sequences, labels, scaler = get_batch_sequences(df, df_stats)

print(f'사용 변수 수 : {len(ONLINE_VARS)}')
print(f'전체 배치 수 : {len(sequences)}')
print(f'Fault 배치  : {sum(v for v in labels.values())}개')
print(f'Normal 배치 : {sum(1 for v in labels.values() if v == 0)}개')

In [ ]:
X_train, X_test, y_test, batch_ids = train_test_split(sequences, labels)

train_windows = make_windows(X_train, WINDOW_SIZE, STRIDE_TR)

print(f'학습 윈도우 수  : {len(train_windows)}')
print(f'윈도우 shape   : {train_windows.shape}  (windows, timesteps, features)')
print(f'테스트 배치 수  : {len(X_test)}')

## 3. 기준선 — MSPC (PCA 기반 SPE)

In [ ]:
train_flat = np.vstack(X_train)

n_components = 10
pca = PCA(n_components=n_components)
pca.fit(train_flat)

explained = pca.explained_variance_ratio_.cumsum()
print(f'PC {n_components}개 누적 분산 설명력: {explained[-1]*100:.1f}%')

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(1, n_components + 1), explained * 100, 'o-', color='steelblue')
ax.axhline(95, color='red', linestyle='--', linewidth=0.8, label='95%')
ax.set_xlabel('Number of Principal Components')
ax.set_ylabel('Cumulative Explained Variance (%)')
ax.set_title('PCA — Cumulative Explained Variance')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def pca_anomaly_scores(pca, sequences):
    scores = []
    for seq in sequences:
        projected     = pca.transform(seq)
        reconstructed = pca.inverse_transform(projected)
        spe = np.mean(np.sum((seq - reconstructed) ** 2, axis=1))
        scores.append(spe)
    return scores

mspc_scores = pca_anomaly_scores(pca, X_test)
mspc_auroc  = roc_auc_score(y_test, mspc_scores)
mspc_auprc  = average_precision_score(y_test, mspc_scores)

print(f'[MSPC Baseline]  AUROC: {mspc_auroc:.4f}  |  AUPRC: {mspc_auprc:.4f}')

## 4. LSTM Autoencoder 학습

In [ ]:
N_FEATURES = len(ONLINE_VARS)

model = LSTMAutoencoder(
    n_features = N_FEATURES,
    hidden_dim = cfg['model']['hidden_dim'],
    latent_dim = cfg['model']['latent_dim'],
    seq_len    = WINDOW_SIZE,
    n_layers   = cfg['model']['n_layers'],
)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\n총 파라미터 수: {total_params:,}')

In [ ]:
train_config = {
    'epochs':     cfg['training']['epochs'],
    'batch_size': cfg['training']['batch_size'],
    'lr':         cfg['training']['lr'],
}

history = train(model, train_windows, train_config, device=DEVICE)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(history, color='steelblue')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('LSTM-AE Training Loss')
plt.tight_layout()
plt.savefig(RESULT_DIR / 'figures' / 'training_loss.png', dpi=150)
plt.show()

In [ ]:
torch.save(model.state_dict(), RESULT_DIR / 'models' / 'lstm_ae.pth')
print('모델 저장 완료: results/models/lstm_ae.pth')

## 5. 이상 점수 산출 및 평가

In [ ]:
result = evaluate(model, X_test, y_test, WINDOW_SIZE, STRIDE_EV, device=DEVICE)

lstm_scores = result['scores']
lstm_auroc  = result['auroc']
lstm_auprc  = result['auprc']

print(f'[LSTM-AE]  AUROC: {lstm_auroc:.4f}  |  AUPRC: {lstm_auprc:.4f}')

## 6. 결과 비교 및 시각화

In [ ]:
results_df = pd.DataFrame({
    'Method': ['MSPC (PCA-SPE)', 'LSTM-AE'],
    'AUROC':  [mspc_auroc, lstm_auroc],
    'AUPRC':  [mspc_auprc, lstm_auprc],
}).set_index('Method').round(4)

print(results_df.to_string())
results_df.to_csv(RESULT_DIR / 'logs' / 'comparison.csv')

In [ ]:
mspc_fpr, mspc_tpr, _ = roc_curve(y_test, mspc_scores)
lstm_fpr, lstm_tpr    = result['fpr'], result['tpr']

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(mspc_fpr, mspc_tpr, label=f'MSPC  (AUROC={mspc_auroc:.3f})', color='#E8644C')
ax.plot(lstm_fpr, lstm_tpr, label=f'LSTM-AE (AUROC={lstm_auroc:.3f})', color='#4C9BE8')
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison')
ax.legend()
plt.tight_layout()
plt.savefig(RESULT_DIR / 'figures' / 'roc_curve.png', dpi=150)
plt.show()

In [ ]:
colors = ['#E8644C' if y == 1 else '#4C9BE8' for y in y_test]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, scores, title in zip(
    axes,
    [mspc_scores, lstm_scores],
    ['MSPC (SPE)', 'LSTM-AE Reconstruction Error']
):
    ax.bar(range(len(scores)), scores, color=colors, alpha=0.8)
    ax.set_xlabel('Batch Index')
    ax.set_ylabel('Anomaly Score')
    ax.set_title(title)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E8644C', label='Fault'),
    Patch(facecolor='#4C9BE8', label='Normal'),
]
axes[0].legend(handles=legend_elements)

plt.tight_layout()
plt.savefig(RESULT_DIR / 'figures' / 'anomaly_scores.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, scores, title in zip(
    axes,
    [mspc_scores, lstm_scores],
    ['MSPC (SPE)', 'LSTM-AE']
):
    normal_s = [s for s, y in zip(scores, y_test) if y == 0]
    fault_s  = [s for s, y in zip(scores, y_test) if y == 1]
    ax.boxplot([normal_s, fault_s], labels=['Normal', 'Fault'],
               patch_artist=True,
               boxprops=dict(facecolor='#4C9BE8', alpha=0.6))
    ax.set_ylabel('Anomaly Score')
    ax.set_title(title)

plt.tight_layout()
plt.savefig(RESULT_DIR / 'figures' / 'score_boxplot.png', dpi=150)
plt.show()

## 실험 결과 메모

### 관찰 사항 (실험 후 직접 기록)

| 항목 | MSPC | LSTM-AE |
|------|------|---------|
| AUROC | - | - |
| AUPRC | - | - |
| 강점 | | |
| 약점 | | |

### 다음 실험 아이디어
- [ ] 주성분 수(n_components) 변경 실험
- [ ] LSTM-AE hidden_dim / latent_dim 하이퍼파라미터 튜닝
- [ ] Transformer-AE 구조 비교
- [ ] 윈도우 크기(window_size) sensitivity 분석